# NB07 — Temporal Signal Stability

**Physics rationale:** bSSFP banding flickers over cardiac phases, degrading CINE quality.
Since our DICOMs are single-frame (one per offset), we use the **sweep neighbourhood** as a proxy.

**Metrics:** `temporal_std_sweep` (3-image sliding window std), `local_signal_drop` (dip vs neighbours),
`pixelwise_temporal_var_myo` (pixel std across all 13 offsets in myo region)

**Selector:** `argmin(temporal_std_sweep)` vs GT

**Runs on:** Local CPU

## 0. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pydicom', 'numpy', 'pandas', 'matplotlib', 'Pillow', 'openpyxl', 'scipy'])
print('Dependencies OK')

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
%matplotlib inline

DATA_ROOT  = r'E:/CINE-Test1/Dr-Dataset/1st_cohort_SAX'
GT_PATH    = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
CACHE_PATH = 'outputs/seg_masks_cache.pkl'
OUT_DIR    = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
EXCLUDE    = ['Patient-KP']
OFFSETS_13 = [-150 + i*25 for i in range(13)]
print('Config OK')

In [ ]:
from utils.dicom_loader import load_all_patients
from utils.gt_loader import load_gt
from utils.seg_wrapper import load_seg_cache

all_data  = load_all_patients(DATA_ROOT, verbose=False)
gt_dict   = load_gt(GT_PATH)
seg_cache = load_seg_cache(CACHE_PATH)
print(f'Loaded {len(all_data)} patients')

## 1. Compute Temporal Metrics

In [ ]:
def align_mask(mask, target_shape):
    th, tw = target_shape[:2]
    mh, mw = mask.shape[:2]
    if (mh, mw) == (th, tw): return mask.astype(bool)
    if (mh, mw) == (tw, th): return mask.T.astype(bool)
    from PIL import Image as PILImage
    pil = PILImage.fromarray(mask.astype(np.uint8) * 255)
    pil = pil.resize((tw, th), PILImage.NEAREST)
    return (np.asarray(pil) > 127)

records = []
for pid in sorted(all_data):
    if pid in EXCLUDE: continue
    data = all_data[pid]
    seg  = seg_cache.get(pid)
    if seg is None: continue

    imgs    = data['images']
    offsets = data['offsets_hz']
    n = len(imgs)

    # Mean myo signal per offset
    mu_myo_list = []
    stack_list  = []
    for img in imgs:
        gray = img.astype(np.float32)
        if gray.max() > 1.0: gray /= gray.max()
        myo_m = align_mask(seg.myocardium_mask, gray.shape)
        myo_px = gray[myo_m]
        mu_myo_list.append(float(myo_px.mean()) if myo_px.size > 0 else 0.0)
        stack_list.append(gray)

    mu_arr = np.array(mu_myo_list)

    # Pixel-wise std across all offsets in myo region
    stack   = np.stack(stack_list, axis=0)
    myo_m0  = align_mask(seg.myocardium_mask, stack.shape[1:])
    pxl_std_all = stack.std(axis=0)
    pixelwise_temporal_var_myo = float(pxl_std_all[myo_m0].mean()) if myo_m0.any() else 0.0

    for i in range(n):
        window_idxs = [j for j in [i-1, i, i+1] if 0 <= j < n]
        window_vals = mu_arr[window_idxs]
        temporal_std_sweep = float(window_vals.std())
        temporal_cv_sweep  = temporal_std_sweep / max(float(window_vals.mean()), 1e-6)
        prev_val = mu_arr[i-1] if i > 0 else mu_arr[i]
        next_val = mu_arr[i+1] if i < n-1 else mu_arr[i]
        local_signal_drop = max(0.0, float((prev_val + next_val) / 2 - mu_arr[i]))

        records.append({
            'patient_id':              pid,
            'image_no':                i + 1,
            'offset_hz':               offsets[i],
            'mu_myo':                  round(mu_arr[i], 6),
            'temporal_std_sweep':      round(temporal_std_sweep, 6),
            'temporal_cv_sweep':       round(temporal_cv_sweep, 6),
            'local_signal_drop':       round(local_signal_drop, 6),
            'pixelwise_temporal_var_myo': round(pixelwise_temporal_var_myo, 6),
        })

df = pd.DataFrame(records)
df.to_csv(f'{OUT_DIR}/temporal_metrics.csv', index=False)
print(f'Computed {len(df)} rows. Saved temporal_metrics.csv')
df.head(13)

## 2. Pixelwise Std Maps (demo) & Metric Curves

In [ ]:
import matplotlib.pyplot as plt

DEMO_PIDS = ['Patient-BEV', 'Patient-INA', 'Patient-DEM']
fig, axes = plt.subplots(2, len(DEMO_PIDS), figsize=(len(DEMO_PIDS)*4, 8))
fig.patch.set_facecolor('#0f0f1a')

for col, pid in enumerate(DEMO_PIDS):
    if pid not in all_data: continue
    data   = all_data[pid]
    gt_no  = gt_dict[pid]['image_no']
    gt_img = data['images'][gt_no-1].astype(np.float32)
    if gt_img.max() > 1.0: gt_img /= gt_img.max()

    stack   = np.stack([img.astype(np.float32) / max(img.max(), 1e-6) for img in data['images']], axis=0)
    pxl_std = stack.std(axis=0)

    axes[0, col].imshow(gt_img, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'{pid}\nGT: {data["offsets_hz"][gt_no-1]:+.0f} Hz', fontsize=8, color='lime')
    axes[0, col].axis('off')
    im = axes[1, col].imshow(pxl_std, cmap='hot')
    plt.colorbar(im, ax=axes[1, col], fraction=0.046)
    axes[1, col].set_title('Pixel Std across 13 offsets', fontsize=7.5, color='white')
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/temporal_pxl_std_demo.png', bbox_inches='tight', dpi=120)
plt.show()

# Metric curves
from utils.vis_utils import plot_metric_curves
for metric_col, title in [
    ('temporal_std_sweep', 'Sweep-Window Std (3-img, lower = stable)'),
    ('local_signal_drop',  'Local Signal Dip (higher = band at this offset)'),
]:
    metric_dict = {
        pid: df[df['patient_id']==pid].sort_values('image_no')[metric_col].tolist()
        for pid in sorted(df['patient_id'].unique())
    }
    fig = plot_metric_curves(
        metric_dict=metric_dict, offsets_hz=OFFSETS_13, gt_dict=gt_dict, title=title,
        save_path=f'{OUT_DIR}/temporal_{metric_col}_curves.png',
    )
    plt.show()

## 3. Accuracy Evaluation

In [ ]:
from utils.gt_loader import evaluate_predictions
from utils.vis_utils import summary_text

for selector_col, selector_name, argfn in [
    ('temporal_std_sweep', 'argmin(temporal_std_sweep)', np.argmin),
    ('local_signal_drop',  'argmin(local_signal_drop)',  np.argmin),
]:
    pred_dict = {}
    for pid in df['patient_id'].unique():
        sub = df[df['patient_id']==pid].sort_values('image_no')
        pred_dict[pid] = int(argfn(sub[selector_col].values)) + 1

    res = evaluate_predictions(pred_dict, gt_dict, exclude=EXCLUDE)
    print(summary_text(selector_name, res['top1'], res['pm1'], res['mae_hz'], res['n_valid']))

print('NB07 complete.')
print('Next: run nb08_cnn_baseline.ipynb on Kaggle GPU, then nb09_multimetric_scoring.ipynb')